In [1]:
import sys, socket, os
from pathlib import Path
protected = [Path.cwd() / 'data', Path.cwd() / 'upstream']
def audit_raw_writes(event, args):
    if event == 'open' and isinstance(args[0], (str, bytes, os.PathLike)):
        path = Path(os.fsdecode(args[0])).resolve()
        mode, flags = args[1], args[2]
        write = (isinstance(mode, str) and any(c in mode for c in 'wax+')) or (isinstance(flags, int) and flags & (os.O_WRONLY | os.O_RDWR | os.O_CREAT | os.O_TRUNC))
        if write and any(path == p or path.is_relative_to(p) for p in protected):
            raise RuntimeError('Protected raw write attempted: ' + str(path))
def no_network(*args, **kwargs):
    raise RuntimeError('Network disabled for inspection verification')
sys.dont_write_bytecode = True
sys.addaudithook(audit_raw_writes)
socket.socket.connect = no_network
socket.create_connection = no_network


# データとtoken数の目視確認

最初のconfiguration cellを変更し、**Restart Kernel and Run All Cells** で上から実行してください。
既存のraw reader・inventory・preprocessing・token counterを呼ぶ閲覧用Notebookです。
data/の取得物、既存CSV、batch結果には書き込みません。OpenAI API・LLM評価・simulationは実行しません。

一覧は公式20 task＋外部5候補。Baiduは未取得、公式9 taskはrawのみで元promptは未対応です。
Notebookの保存出力は実行時の設定に対応するsnapshotなので、設定変更後は全セルを再実行してください。


In [2]:
DATASET = "MountainCar-v0"
EPISODE = 0
QUERY_INDEX = 10
HISTORY_SIZE = 1
HISTORY_SIZES = [0, 1, 2, 3, 5]

# None / None: 選択taskの既存LLM-X既定質問を使用。
# 指定する場合は両方変更。例: "next-action", "next_action_prediction"
METRIC = None
QUESTION_NAME = None
MODEL = "gpt-4o"

# 外部trajectory: SEQUENCE_IDとQUERY_INDEX。static: ROW_ID（全reader単位の通し番号）。
SEQUENCE_ID = None
ROW_ID = None

# 外部token表示は明示的に指定した場合だけ。Noneならraw inspectionのみ。
# 例: "configs/preprocessing/f16capstone_default.yaml"
PREPROCESSING_CONFIG = None
EXTERNAL_FULL_DATASET = False  # Trueで明示設定scope全体。既定は選択sequenceのみ。
MAX_EXTERNAL_QUERIES = 100_000 # 上限を超える計測は実行しない（sampleを全件に見せない）。

DATASET = 'f16capstone'
SEQUENCE_ID = 0

In [3]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

# workspace直下／notebooks/のどちらから起動しても同じ既存moduleをimport。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "dataset_adapters").is_dir() and (p / "configs/sources.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("workspace直下またはnotebooks/から起動してください。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dataset_adapters.registry import get_adapter, EXTERNAL
from tools.notebook_inspection import (
    dataset_table, select_record, inspect_selected_file, preview_raw,
    preprocessing_for_inspection, selected_query, token_record,
    dataset_token_table, table_statistics, read_batch_summary,
)
from tools.count_input_tokens import resolve_encoding

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)
print("Workspace:", ROOT)
print("Read-only inspection. No API client or API request.")


Workspace: /Users/cls-lab/Git/Matsuo/mental-modeling-openai
Read-only inspection. No API client or API request.


## 1. 全dataset一覧

既存inventoryを使うlist_datasets.pyの共通処理から作成します。外部の固定state/action/reward shapeは未定義（null）で、欠損値を0にしません。外部record数は異種stream・duplicate export等を含むreader件数です。


In [4]:
datasets = dataset_table()
overview_columns = [
    "dataset_id", "category", "status", "files", "sequences", "records",
    "state_shape", "state_dtype", "action_shape", "action_dtype",
    "reward_shape", "reward_dtype", "sequential", "prompt_mode",
]
with pd.option_context("display.max_rows", None):
    display(datasets[overview_columns])
print(f"{len(datasets)} datasets; Baidu: NOT_ACQUIRED")


,dataset_id,category,status,files,sequences,records,state_shape,state_dtype,action_shape,action_dtype,reward_shape,reward_dtype,sequential,prompt_mode
0,Acrobot-v1,official,acquired,5,5.0,399.0,"[""T"", 1, 6]",float32,"[""T"", 1]",int64,"[""T"", 1]",float64,True,original_llmx
1,BipedalWalker-v3,official,acquired,2,2.0,144.0,"[""T"", 1, 24]",float32,"[""T"", 1, 4]",float32,"[""T"", 1]",float64,True,unsupported
2,CartPole-v1,official,acquired,5,5.0,2500.0,"[""T"", 1, 4]",float32,"[""T"", 1]",int64,"[""T"", 1]",float64,True,original_llmx
3,FetchPickAndPlace-v2,official,acquired,10,10.0,500.0,"[""T"", 1, 26]",float64,"[""T"", 1, 4]",float32,"[""T"", 1]",float32,True,original_llmx
4,FetchPush-v2,official,acquired,10,10.0,500.0,"[""T"", 1, 26]",float64,"[""T"", 1, 4]",float32,"[""T"", 1]",float32,True,original_llmx
5,FetchSlide-v2,official,acquired,10,10.0,500.0,"[""T"", 1, 26]",float64,"[""T"", 1, 4]",float32,"[""T"", 1]",float32,True,original_llmx
6,HalfCheetah-v4,official,acquired,3,3.0,3000.0,"[""T"", 1, 17]",float64,"[""T"", 1, 6]",float32,"[""T"", 1]",float64,True,unsupported
7,InvertedDoublePendulum-v4,official,acquired,10,10.0,538.0,"[""T"", 1, 11]",float64,"[""T"", 1, 1]",float32,"[""T"", 1]",float64,True,original_llmx
8,InvertedPendulum-v4,official,acquired,5,5.0,1419.0,"[""T"", 1, 4]",float64,"[""T"", 1, 1]",float32,"[""T"", 1]",float64,True,original_llmx
9,LunarLander-v2,official,acquired,3,3.0,744.0,"[""T"", 1, 8]",float32,"[""T"", 1]",int64,"[""T"", 1]",float64,True,original_llmx


25 datasets; Baidu: NOT_ACQUIRED


## 2. 選択datasetのschema — RAW DATA

選択元ファイルを既存inventoryコードで再読込し、保存済みinventoryのshape/dtypeと並べます。
shapeのsingleton次元を省略しません。CSVにはネイティブ数値dtypeがないため、schemaはinventory readerの型解釈、raw valueはreaderが保持した元の文字列として区別します。

TrajAirの配布元processed TXTは取得物の一部で、本projectがrawへ加工保存したものではありません。
F16のopaque MATLAB object・モデル等はraw adapter非対応のままです。


In [5]:
adapter = get_adapter(DATASET)
description = adapter.describe()
display(pd.DataFrame([
    {"item": "Dataset", "value": DATASET},
    {"item": "Source", "value": description["source"]},
    {"item": "Format", "value": description["format"]},
    {"item": "Files", "value": description["number_of_files"]},
    {"item": "Episodes / trajectories", "value": description["number_of_sequences"]},
    {"item": "Timesteps / rows (reader scope)", "value": description["number_of_records"]},
]))
sequence, raw_record = select_record(
    adapter, episode=EPISODE, sequence_id=SEQUENCE_ID, row_id=ROW_ID, index=QUERY_INDEX
)
schema_table = pd.DataFrame()
if sequence is None:
    print("NOT_ACQUIRED: raw data inspection unavailable.")
else:
    print("Source file:", sequence.source_file)
    print("Sequence:", sequence.sequence_id, "key:", sequence.key)
    print("Selected reader-unit records:", sequence.length)
    schema_table, file_counts, actual_schema = inspect_selected_file(adapter, sequence)
    print("Actual file read vs saved inventory (file-level schema):")
    with pd.option_context("display.max_rows", None):
        display(schema_table)
    display(file_counts)
    if description.get("unreadable_variables"):
        print("Opaque MATLAB variables are inventory-only; not converted to trajectories.")
        display(pd.DataFrame(description["unreadable_variables"])[["source_file", "variable", "reason"]])


,item,value
0,Dataset,f16capstone
1,Source,{'commit': 'acedfe4b2401b600a89f14f771dbcdf88c...
2,Format,"[.csv, .f3d, .f3z, .heic, .jpeg, .jpg, .json, ..."
3,Files,210
4,Episodes / trajectories,338
5,Timesteps / rows (reader scope),836572


Source file: data/candidate_datasets/f16capstone/repository/Mathematical Model/System Identification/20_05_2021__18_44_27.csv
Sequence: 0 key: repository/Mathematical Model/System Identification/20_05_2021__18_44_27.csv
Selected reader-unit records: 18980
Actual file read vs saved inventory (file-level schema):


,name,shape,dtype,ndim,shape_inventory,dtype_inventory,ndim_inventory,matches_inventory
0,m-accelerationBodyAX-MPS2,[18980],float64,1,[18980],float64,1,True
1,m-accelerationBodyAY-MPS2,[18980],float64,1,[18980],float64,1,True
2,m-accelerationBodyAZ-MPS2,[18980],float64,1,[18980],float64,1,True
3,m-accelerationWorldAX-MPS2,[18980],float64,1,[18980],float64,1,True
4,m-accelerationWorldAY-MPS2,[18980],float64,1,[18980],float64,1,True
5,m-accelerationWorldAZ-MPS2,[18980],float64,1,[18980],float64,1,True
6,m-aircraftPositionX-MTR,[18980],float64,1,[18980],float64,1,True
7,m-aircraftPositionY-MTR,[18980],float64,1,[18980],float64,1,True
8,m-airspeed-MPS,[18980],float64,1,[18980],float64,1,True
9,m-altitudeAGL-MTR,[18980],float64,1,[18980],float64,1,True


,measure,actual,inventory,matches
0,rows,18980,18980,True
1,columns,55,55,True


Opaque MATLAB variables are inventory-only; not converted to trajectories.


,source_file,variable,reason
0,data/candidate_datasets/f16capstone/repository...,vals,"opaque MATLAB object, internal rows unknown"
1,data/candidate_datasets/f16capstone/repository...,chirp,"opaque MATLAB object, internal rows unknown"
2,data/candidate_datasets/f16capstone/repository...,out,"opaque MATLAB object, internal rows unknown"
3,data/candidate_datasets/f16capstone/repository...,vals,"opaque MATLAB object, internal rows unknown"


## 3. 実際のraw value — RAW DATA

先頭5 recordと指定番号を表示します。公式はstates[:5] / actions[:5] / rewards[:5]を含む全arrayを表示。
episodic_returnはepisode全体の値として別扱いです。外部では元columnをそのまま表示し、state/action/rewardへ変換しません。

外部の別ファイルを選ぶには `adapter.sequences()` のsequence_idをconfigurationへ設定できます。


In [6]:
raw_preview = None
if sequence is not None:
    raw_preview = preview_raw(adapter, sequence, count=5)
    if isinstance(raw_preview, dict):
        for name, values in raw_preview.items():
            print(f"\n{name} (episode scope)" if name == "episodic_return" else f"\n{name}[:5]")
            print(values)
    else:
        display(raw_preview)
    print("\nSELECTED RAW RECORD")
    print("episode =", raw_record["episode_id"], "sequence =", raw_record["sequence_id"],
          "row =", raw_record.get("row_id"), "index =", raw_record["index"])
    print("Source:", raw_record["source_file"])
    print("SHA256:", raw_record["source_file_sha256"])
    # JSON表示なので指定recordの値をDataFrameの省略表示に隠しません。
    print(json.dumps(raw_record["fields"], ensure_ascii=False, indent=2))
else:
    print("Raw value: N/A (not acquired)")


,index,m-airspeed-MPS,m-altitudeASL-MTR,m-altitudeAGL-MTR,m-groundspeed-MPS,m-pitchRate-DEGpSEC,m-rollRate-DEGpSEC,m-yawRate-DEGpSEC,m-azimuth-DEG,m-inclination-DEG,m-roll-DEG,m-aircraftPositionX-MTR,m-aircraftPositionY-MTR,m-velocityWorldU-MPS,m-velocityWorldV-MPS,m-velocityWorldW-MPS,m-velocityBodyU-MPS,m-velocityBodyV-MPS,m-velocityBodyW-MPS,m-accelerationWorldAX-MPS2,m-accelerationWorldAY-MPS2,m-accelerationWorldAZ-MPS2,m-accelerationBodyAX-MPS2,m-accelerationBodyAY-MPS2,m-accelerationBodyAZ-MPS2,m-windX-MPS,m-windY-MPS,m-windZ-MPS,m-propRPM,m-heliMainRotorRPM,m-batteryVoltage-VOLTS,m-batteryCurrentDraw-AMPS,m-batteryRemainingCapacity-MAH,m-fuelRemaining-OZ,m-isLocked,m-hasLostComponents,m-anEngineIsRunning,m-isTouchingGround,m-currentPhysicsTime-SEC,m-currentPhysicsSpeedMultiplier,m-orientationQuaternion-X,m-orientationQuaternion-Y,m-orientationQuaternion-Z,m-orientationQuaternion-W,rc_channel_0,rc_channel_1,rc_channel_2,rc_channel_3,rc_channel_4,rc_channel_5,rc_channel_6,rc_channel_7,rc_channel_8,rc_channel_9,rc_channel_10,rc_channel_11
0,0,0.00555139230374887,0.07121964535518455,0.07121964535518455,0.005146647335692136,0.10493183027173814,0.30565767204302574,-0.0816177945729919,0.2078419029712677,-0.30350154638290405,0.3856721818447113,63.649269104003906,64.29619598388672,0.004775706212967634,0.001918491325341165,0.002080859849229455,0.004779670387506485,0.0019495952874422073,0.0020424933172762394,2.1586103439331055,-0.00783457886427641,9.818084716796875,1.9995925463736057,-0.46166473627090454,-8.997926712036133,0,0,0,20922.49609375,-1,24.23939634633473,7.371776103973389,3603.445068359375,-1,False,False,True,True,144.2885386685375,1,-0.002654633019119501,0.0033607976511120796,0.0018048312049359083,0.9999891519546509,0.5,0.5,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0
1,1,0.008321079895161251,0.07123535115543511,0.07123535115543511,0.0072152868035204034,1.1859361392012033,0.26605551703818264,0.26381646909339906,0.20879662036895752,-0.29925259947776794,0.38770729303359985,63.649269104003906,64.2962875366211,0.006834238301962614,-0.0023137740790843964,0.004144877195358276,0.006864179857075214,-0.002260996028780937,0.00412447564303875,2.1578822135925293,-0.007843761704862118,9.817922592163086,-0.3426456041634083,-0.3007046729326248,-7.675017833709717,0,0,0,20922.4765625,-1,24.239255977563616,7.371720790863037,3603.408935546875,-1,False,False,True,True,144.30615586834028,1,-0.0026176143437623978,0.0033786026760935783,0.001813241047784686,0.9999892711639404,0.5,0.5,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0
2,2,0.006045393194438805,0.07124075144360686,0.07124075144360686,0.00557430662603162,-0.43871081698648595,-0.12442270781312459,0.5223823372723473,0.2120172083377838,-0.29924145340919495,0.38748010993003845,63.64924621582031,64.29634094238281,0.004034258890897036,-0.003846771316602826,0.0023396334145218134,0.004060629289597273,-0.0038160497788339853,0.0023443177342414856,2.1584577560424805,-0.007935664616525173,9.817926406860352,-0.16824742034077644,-0.19089962542057037,-8.919336318969727,0,0,0,20922.390625,-1,24.23918266393794,7.371674537658691,3603.39013671875,-1,False,False,True,True,144.31530706817284,1,-0.00261760875582695,0.003376547247171402,0.0018413515063002706,0.9999890327453613,0.5,0.5,0.5,0.5,0.0,0.0,0.5,0.5,0.0,0.0,0.0,0.0
3,3,0.005588964099998627,0.07124251369071422,0.07124251369071422,0.005208870706421626,-0.3802713214501523,-0.18094957706409787,0.7141653408172033,0.2151716947555542,-0.2999512851238251,0.38710692524909973,63.64922332763672,64.29635620117188,0.002126024104654789,-0.004755245056003332,0.0020258789882063866,0.0021544434130191803,-0.004733507055789232,0.0020466549322009087,2.1584348678588867,-0.008046566508710384,9.817947387695312,-0.12355591356754303,-0.1423710659146309,-8.916732788085938,0,0,0,20922.33203125,-1,24.239137960507648,7.3716349601745605,3603.378662109375,-1,False,False,True,True,144.32083216821775,1,-0.0026238898281008005,0.0033732065930962563,0.001868866616860032,0.9999890923500061,0.5,0.


SELECTED RAW RECORD
episode = None sequence = 0 row = None index = 10
Source: data/candidate_datasets/f16capstone/repository/Mathematical Model/System Identification/20_05_2021__18_44_27.csv
SHA256: 60296b42a9a30f470e75ad7402dd7c04a8cc85ae7738fa57b5332df9e384be4f
{
  "m-airspeed-MPS": {
    "value": "0.0076530540022480085",
    "dtype": "<U21",
    "shape": []
  },
  "m-altitudeASL-MTR": {
    "value": "0.07124708281594616",
    "dtype": "<U19",
    "shape": []
  },
  "m-altitudeAGL-MTR": {
    "value": "0.07124708281594616",
    "dtype": "<U19",
    "shape": []
  },
  "m-groundspeed-MPS": {
    "value": "0.0020938503665184523",
    "dtype": "<U21",
    "shape": []
  },
  "m-pitchRate-DEGpSEC": {
    "value": "2.2241799044372783",
    "dtype": "<U18",
    "shape": []
  },
  "m-rollRate-DEGpSEC": {
    "value": "-0.28021970109307404",
    "dtype": "<U20",
    "shape": []
  },
  "m-yawRate-DEGpSEC": {
    "value": "0.21845430381995357",
    "dtype": "<U19",
    "shape": []
  },
  "m-azi

## 4. MODEL INPUT AFTER PREPROCESSING

ここからraw表示とは別です。公式対応taskはpreprocessing/llmx_original.pyを通し、元の
Episode / history_text / render_question / system_prompt / user_promptをそのまま使います。
未対応taskのpromptは作りません。

外部はPREPROCESSING_CONFIGを明示した場合だけ、その既存baselineを使います。
このpreprocessingはraw datasetそのものではなく、本projectで定義したLLM入力用変換です。
staticはH=0のみ。存在しないaction/rewardを生成しません。


In [7]:
config, preprocessing_status = preprocessing_for_inspection(DATASET, PREPROCESSING_CONFIG)
print("Raw data inspection:", "available" if sequence is not None else "unavailable")
print(preprocessing_status)
query = None
if config is not None and sequence is not None:
    display(config)
    try:
        query = selected_query(adapter, sequence, raw_record["index"], config,
                               HISTORY_SIZE, METRIC, QUESTION_NAME)
    except (ValueError, IndexError) as exc:
        print("Selected model input: N/A —", exc)
    if query is not None:
        print("Prompt mode:", config["prompt_mode"])
        print("Metric / question:", query.metric, "/", query.question_name)
        print("Effective history size:", query.history_size)
        print("History interval [start, end):", query.history_start, query.history_end)
        print("\nHISTORY REPRESENTATION\n")
        print(query.history_text)
else:
    print("Token count: N/A")


Raw data inspection: available
Model-input preprocessing: not defined (PREPROCESSING_CONFIG is not selected)
Token count: N/A


## 5. 実際のprompt全文

以下が実際のSYSTEM PROMPTとUSER PROMPTです。外部設定時は元LLM-Xではなくexternal baselineと表示します。文字列を組み立てるだけで送信はしません。


In [8]:
if query is not None:
    print("Original LLM-X" if config["prompt_mode"] == "original_llmx" else "EXTERNAL BASELINE")
    print("\nSYSTEM PROMPT\n")
    print(query.system_prompt)
    print("\nUSER PROMPT\n")
    print(query.user_prompt)
else:
    print("Prompt: N/A")


Prompt: N/A


## 6. 選択queryのtoken数

既存count_input_tokens.count_queryを使用。contentは実文字列のlocal tokenize結果です。
estimated_api_input_tokensは既存CLIと同じcontent + 2×3 + 3の概算で、server usageではありません。
history/questionを別々にtokenizeした値はBPE境界のためuser_tokensへ単純加算できません。
未知のmodel名ではfallbackを明示します。tokenizer未キャッシュ時だけ公開tokenizer assetの取得が必要になる場合があります（inference APIではありません）。


In [9]:
selected_tokens = None
if query is not None:
    encoding, fallback_used = resolve_encoding(MODEL)
    print("Model:", MODEL, "\nTokenizer:", encoding.name)
    print("Fallback tokenizer:", fallback_used)
    if fallback_used:
        print("WARNING: unknown model; fallback tokenizer is being used.")
    selected_tokens = token_record(query, sequence, config, encoding, fallback_used, MODEL)
    token_columns = ["system_tokens", "history_tokens", "question_tokens", "user_tokens",
                     "total_content_tokens", "estimated_api_input_tokens"]
    display(pd.DataFrame([{"component": key, "tokens": selected_tokens[key]} for key in token_columns]))
    print("Preprocessing SHA256:", selected_tokens["preprocessing_config_sha256"])
else:
    print("Token count: N/A")


Token count: N/A


## 7. データ番号ごとのtoken数

公式は選択datasetの**全episode・全valid query**を、現在のH / metric / questionで再計測します。
全行はquery_tokens DataFrameに保持し、長い表の画面表示は先頭・末尾に省略されます。
外部は既定で選択sequenceのみ。全設定scopeはEXTERNAL_FULL_DATASET=Trueで明示し、MAX_EXTERNAL_QUERIESを超えた場合は実行しません。自動samplingはしません。


In [10]:
query_tokens, count_scope = dataset_token_table(
    adapter, config, HISTORY_SIZE, MODEL, METRIC, QUESTION_NAME,
    external_sequence=sequence.sequence_id if sequence else None,
    external_full_dataset=EXTERNAL_FULL_DATASET,
    max_external_queries=MAX_EXTERNAL_QUERIES,
)
print("Scope:", count_scope)
print("Computed query rows:", len(query_tokens))
query_columns = ["episode_id", "sequence_id", "query_index", "history_start", "history_end",
                 "input_tokens", "source_file"]
if not query_tokens.empty:
    display(query_tokens[query_columns])
    if query_tokens["fallback_encoding_used"].any():
        print("WARNING: fallback tokenizer used:", query_tokens["tokenizer"].unique())


Scope: Token count: N/A
Computed query rows: 0


## 8. token統計・上位5件／下位5件

既存token_statisticsを利用。stdは母標準偏差(ddof=0)、p95はnearest-rankです。
input_tokensは上記のAPI入力概算を指します。


In [11]:
statistics_table = pd.DataFrame()
if not query_tokens.empty:
    statistics_table = pd.DataFrame([table_statistics(query_tokens)])
    display(statistics_table)
    print("Largest 5 queries")
    display(query_tokens.nlargest(5, "input_tokens")[query_columns])
    print("Smallest 5 queries")
    display(query_tokens.nsmallest(5, "input_tokens")[query_columns])
else:
    print("Statistics: N/A")


Statistics: N/A


## 9. token分布

In [12]:
if not query_tokens.empty:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(query_tokens["input_tokens"], bins=30)
    ax.set_xlabel("Estimated API input tokens")
    ax.set_ylabel("Query count")
    ax.set_title(f"{DATASET} | H={query_tokens['history_size'].iloc[0]}")
    fig.tight_layout()
    plt.show()
else:
    print("Histogram: N/A")


Histogram: N/A


## 10. history size比較

選択datasetに対してのみ計算します。各Hで有効query集合が変わるため、totalの差と同一queryの増加は別です。
各Hの全行はhistory_tables[H]に保持します。比較にはepisode_id / sequence_id / query_indexでjoinできます。
外部staticはH=0のみ表示します。


In [13]:
history_tables = {}
history_rows = []
if config is not None:
    hs = [0] if config["prompt_mode"] == "external_static" else sorted(set(HISTORY_SIZES))
    for h in hs:
        current_h = 0 if config["prompt_mode"] == "external_static" else HISTORY_SIZE
        if h == current_h:
            table, scope = query_tokens, count_scope
        else:
            table, scope = dataset_token_table(
                adapter, config, h, MODEL, METRIC, QUESTION_NAME,
                external_sequence=sequence.sequence_id if sequence else None,
                external_full_dataset=EXTERNAL_FULL_DATASET,
                max_external_queries=MAX_EXTERNAL_QUERIES,
            )
        history_tables[h] = table
        if table.empty:
            print(f"H={h}: no queries — {scope}")
            continue
        history_rows.append({"history_size": h, **table_statistics(table), "scope": scope})
history_comparison = pd.DataFrame(history_rows)
display(history_comparison if not history_comparison.empty else "History comparison: N/A")


'History comparison: N/A'

## 11. 保存済み全dataset token summary

outputs/token_counts/token_inventory.csvがあれば読み込むだけです。model/tokenizerの保存時情報も表示します。
**この表は現在のconfigurationで再計算した結果ではありません。**
prompt_mode・H・config SHA・全件/sample scopeを維持し、混ぜて合計しません。
CSVがなければbatchを起動せずメッセージだけ表示します。


In [14]:
batch_summary = read_batch_summary()
if batch_summary is None:
    print("Token batch has not been executed yet.")
else:
    metadata_path = ROOT / "outputs/token_counts/summary.json"
    if metadata_path.is_file():
        metadata = json.loads(metadata_path.read_text())
        print("Saved batch model:", metadata.get("model"), "tokenizer:", metadata.get("tokenizer"))
    batch_columns = [
        "dataset", "n_units", "prompt_mode", "history_size", "count_scope",
        "mean_input_tokens", "std_input_tokens", "min_input_tokens", "max_input_tokens",
        "p95_input_tokens", "total_input_tokens", "preprocessing_config_sha256",
    ]
    with pd.option_context("display.max_rows", None):
        display(batch_summary[[c for c in batch_columns if c in batch_summary]])
    print("No batch was started. Existing results were read only.")


Saved batch model: gpt-4o tokenizer: o200k_base


,dataset,n_units,prompt_mode,history_size,count_scope,mean_input_tokens,std_input_tokens,min_input_tokens,max_input_tokens,p95_input_tokens,total_input_tokens,preprocessing_config_sha256
0,Acrobot-v1,394,original_llmx,0,full_selected_scope,829.604061,2.687739,824,851,832,326864,93dfa8c41a457a02bc85c81dcf71a31bf96226ea544bc7...
1,Acrobot-v1,389,original_llmx,1,full_selected_scope,887.868895,3.258492,880,910,890,345381,08dcac0bdc5397de4bf8cc6a1df8833e2094350602bf0b...
2,Acrobot-v1,384,original_llmx,2,full_selected_scope,946.148438,3.760589,938,969,949,363321,e65b1fd900a1a9f5fc12e4409906252cebd7498879e226...
3,Acrobot-v1,379,original_llmx,3,full_selected_scope,1004.432718,4.279495,996,1028,1008,380680,c0c4a33cf5f4d0c0bc4f4b7d57e8227c90e984b61cce88...
4,Acrobot-v1,369,original_llmx,5,full_selected_scope,1120.978320,5.256517,1112,1146,1126,413641,8f9e90a0f3fdba34c0f15aa6a4ae696fa21b4db2805e19...
5,Acrobot-v1,344,original_llmx,10,full_selected_scope,1412.287791,7.274220,1400,1438,1429,485827,aa547e15a1fa5f9d58e077cd5bc5b80fa8ddc06e53de3a...
6,CartPole-v1,2495,original_llmx,0,full_selected_scope,785.440080,2.056203,781,799,787,1959673,93dfa8c41a457a02bc85c81dcf71a31bf96226ea544bc7...
7,CartPole-v1,2490,original_llmx,1,full_selected_scope,830.610040,2.619846,825,845,833,2068219,08dcac0bdc5397de4bf8cc6a1df8833e2094350602bf0b...
8,CartPole-v1,2485,original_llmx,2,full_selected_scope,875.781087,3.118770,868,891,879,2176316,e65b1fd900a1a9f5fc12e4409906252cebd7498879e226...
9,CartPole-v1,2480,original_llmx,3,full_selected_scope,920.946774,3.566928,912,936,925,2283948,c0c4a33cf5f4d0c0bc4f4b7d57e8227c90e984b61cce88...


No batch was started. Existing results were read only.


### 処理の所在

- 一覧: tools/list_datasets.py、保存済みoutputs/dataset_inventory/
- raw読込: dataset_adapters/、再照合: tools/inventory_llmx_data.py / inventory_candidates.py
- prompt: preprocessing/llmx_original.py / prompting.py（Notebook内では再実装しない）
- local token / 統計: tools/count_input_tokens.py
- 閲覧用の表への整形: tools/notebook_inspection.py

参考: [OpenAI GPT-4o](https://developers.openai.com/api/docs/models/gpt-4o)、[Text generation](https://developers.openai.com/api/docs/guides/text)。
ここではモデル推論もAPI呼び出しも行っていません。


In [15]:
assert len(raw_record['fields']) == 55
assert len(raw_preview) == 5
assert schema_table.matches_inventory.all()
assert config is None and query is None
assert query_tokens.empty
print('F16 raw-only assertions passed')

F16 raw-only assertions passed
